In [ ]:
import kagglehub
path = kagglehub.dataset_download("prasad22/healthcare-dataset")
print("Dataset downloaded to:", path)

In [ ]:
import os
import pandas as pd

# List the contents of the downloaded dataset folder
files = os.listdir(path)
print("Files in dataset directory:", files)

# Find and load the main CSV file
csv_files = [f for f in files if f.endswith('.csv')]
if csv_files:
    csv_path = os.path.join(path, csv_files[0])
    df = pd.read_csv(csv_path)
    print(f"\nLoaded {csv_files[0]} successfully!")
    display(df.head())
else:
    print("No CSV files found in the dataset directory.")

In [ ]:
# 1. Clean data: check for null values and duplicate records
print("=== Missing Values Summary ===")
print(df.isnull().sum())

print("\n=== Duplicate Records Summary ===")
duplicates_count = df.duplicated().sum()
print(f"Number of duplicate rows: {duplicates_count}")

# Drop duplicates if any
if duplicates_count > 0:
    df = df.drop_duplicates()
    print("Duplicates removed!")

# Clean string irregularities in the 'Name' column for cleaner data presentation
if 'Name' in df.columns:
    df['Name'] = df['Name'].str.title()

# Convert date columns to datetime objects
if 'Date of Admission' in df.columns:
    df['Date of Admission'] = pd.to_datetime(df['Date of Admission'])
if 'Discharge Date' in df.columns:
    df['Discharge Date'] = pd.to_datetime(df['Discharge Date'])

print("\n=== Dataset Info After Cleaning ===")
print(df.info())

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Set the aesthetic style for the plots
sns.set_theme(style="whitegrid")

# Prepare figure for subplots (1 row, 2 columns)
fig, axes = plt.subplots(1, 2, figsize=(16, 7))

# 2. Line Chart: Average Billing Amount by Age Group
# Let's bin the age into decades to make a smoother line chart
df['Age Group'] = pd.cut(df['Age'], bins=range(10, 101, 10), labels=[f"{i}-{i+9}" for i in range(10, 91, 10)])
age_billing = df.groupby('Age Group', observed=False)['Billing Amount'].mean().reset_index()

sns.lineplot(
    data=age_billing, 
    x='Age Group', 
    y='Billing Amount', 
    marker='o', 
    color='royalblue', 
    linewidth=2.5, 
    ax=axes[0]
)
axes[0].set_title('Average Billing Amount by Age Group', fontsize=14, pad=15)
axes[0].set_xlabel('Age Group', fontsize=12)
axes[0].set_ylabel('Average Billing Amount ($)', fontsize=12)
axes[0].grid(True, linestyle='--', alpha=0.6)

# 3. Doughnut Chart: Distribution of Admission Types
admission_counts = df['Admission Type'].value_counts()
colors = ['#ff9999', '#66b3ff', '#99ff99']

# Draw pie
wedges, texts, autotexts = axes[1].pie(
    admission_counts, 
    labels=admission_counts.index, 
    autopct='%1.1f%%', 
    startangle=90, 
    colors=colors, 
    textprops=dict(color="black", size=12),
    wedgeprops=dict(width=0.4, edgecolor='white') # This width setting creates the 'hole' in the middle (doughnut)
)

# Format the percentages inside the doughnut chart
for autotext in autotexts:
    autotext.set_size(11)
    autotext.set_weight('bold')

axes[1].set_title('Distribution of Admission Types', fontsize=14, pad=15)

plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

# --- 1. Calculate Length of Stay (LOS) in Days ---
df['Length of Stay'] = (df['Discharge Date'] - df['Date of Admission']).dt.days

# Set up a multi-plot figure
fig, axes = plt.subplots(2, 2, figsize=(18, 14))

# Plot A: Average Length of Stay by Medical Condition
los_by_condition = df.groupby('Medical Condition')['Length of Stay'].mean().sort_values(ascending=False).reset_index()
sns.barplot(
    data=los_by_condition,
    x='Length of Stay',
    y='Medical Condition',
    palette='viridis',
    hue='Medical Condition',
    legend=False,
    ax=axes[0, 0]
)
axes[0, 0].set_title('Average Length of Stay (Days) by Medical Condition', fontsize=13)
axes[0, 0].set_xlabel('Average Days')
axes[0, 0].set_ylabel('Medical Condition')

# Plot B: Average Billing Amount by Insurance Provider
billing_by_insurance = df.groupby('Insurance Provider')['Billing Amount'].mean().sort_values(ascending=False).reset_index()
sns.barplot(
    data=billing_by_insurance,
    x='Insurance Provider',
    y='Billing Amount',
    palette='pastel',
    hue='Insurance Provider',
    legend=False,
    ax=axes[0, 1]
)
axes[0, 1].set_title('Average Billing Amount by Insurance Provider', fontsize=13)
axes[0, 1].set_xlabel('Insurance Provider')
axes[0, 1].set_ylabel('Average Bill ($)')

# Plot C: Medication vs. Medical Condition Heatmap
med_condition_pivot = pd.crosstab(df['Medical Condition'], df['Medication'], normalize='index') * 100
sns.heatmap(
    med_condition_pivot,
    annot=True,
    fmt=".1f",
    cmap='YlGnBu',
    cbar_kws={'label': '% of Condition Prescribed'},
    ax=axes[1, 0]
)
axes[1, 0].set_title('Medication Prescription Distribution by Condition (%)', fontsize=13)
axes[1, 0].set_xlabel('Medication')
axes[1, 0].set_ylabel('Medical Condition')

# Plot D: Test Results by Medical Condition
sns.countplot(
    data=df,
    x='Medical Condition',
    hue='Test Results',
    palette='Set2',
    ax=axes[1, 1]
)
axes[1, 1].set_title('Test Results Distribution by Medical Condition', fontsize=13)
axes[1, 1].set_xlabel('Medical Condition')
axes[1, 1].set_ylabel('Count')
axes[1, 1].tick_params(axis='x', rotation=30)

plt.tight_layout()
plt.show()